# 音楽字幕ジェネレーター

音楽ファイルと歌詞テキストから YouTube 用の字幕ファイル（SRT形式）を自動生成します。

**手順**
1. 「セル 1: 環境セットアップ」を実行
2. 「セル 2: ファイルアップロード」で音楽ファイルと歌詞ファイルをアップロード
3. 「セル 3: 設定」でモデルや言語を設定
4. 「セル 4: 字幕生成」を実行 → SRTファイルが自動ダウンロードされます

> **ヒント**: ランタイムの種類を「GPU」に設定すると処理が速くなります（ランタイム → ランタイムのタイプを変更）

In [ ]:
# セル 1: 環境セットアップ
print("ffmpeg をインストール中...")
!apt-get install -y -q ffmpeg

print("stable-whisper をインストール中...")
!pip install -q stable-whisper

print("\n✓ セットアップ完了")

In [ ]:
# セル 2: ファイルアップロード
from google.colab import files

print("音楽ファイルをアップロードしてください（MP3 / WAV）")
uploaded_audio = files.upload()
audio_filename = list(uploaded_audio.keys())[0]
print(f"✓ 音楽ファイル: {audio_filename}")

print("\n歌詞ファイルをアップロードしてください（TXT）")
uploaded_lyrics = files.upload()
lyrics_filename = list(uploaded_lyrics.keys())[0]
print(f"✓ 歌詞ファイル: {lyrics_filename}")

In [ ]:
# セル 3: 設定

# 使用するWhisperモデル
# tiny / base: 高速・低精度（動作確認用）
# small: バランス型（英語向け）
# medium: 高精度（日本語向け）
# large-v3: 最高精度・低速（日本語・精度最優先）
MODEL = "large-v3"

# 言語コード（None で自動検出）
# 日本語: "ja" / 英語: "en" / 韓国語: "ko"
LANGUAGE = "ja"

# 出力ファイル名（None で音楽ファイル名.srt）
OUTPUT_FILENAME = None

print(f"モデル   : {MODEL}")
print(f"言語     : {LANGUAGE or '自動検出'}")

In [ ]:
# セル 4: 字幕生成
from pathlib import Path
import sys


def format_srt_time(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    ms = int(round((seconds % 1) * 1000))
    if ms >= 1000:
        ms, s = 0, s + 1
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def write_srt(segments: list, output_path: Path) -> None:
    with open(output_path, "w", encoding="utf-8") as f:
        idx = 1
        for seg in segments:
            text = seg["text"].strip()
            if not text:
                continue
            f.write(f"{idx}\n{format_srt_time(seg['start'])} --> {format_srt_time(seg['end'])}\n\u266a {text}\n\n")
            idx += 1


def is_suno_tag(line: str) -> bool:
    return line.startswith("[") and line.endswith("]")


def load_lyrics(path: Path) -> list:
    with open(path, "r", encoding="utf-8") as f:
        return [
            line.strip()
            for line in f
            if line.strip() and not line.startswith("#") and not is_suno_tag(line.strip())
        ]


def fix_overlaps(segments: list, min_gap: float = 0.05) -> list:
    for i in range(len(segments) - 1):
        next_start = segments[i + 1]["start"]
        if segments[i]["end"] > next_start - min_gap:
            segments[i]["end"] = max(segments[i]["start"] + 0.1, next_start - min_gap)
    return segments


# 歌詞を読み込む
lyrics_path = Path(lyrics_filename)
lyrics_lines = load_lyrics(lyrics_path)
if not lyrics_lines:
    raise ValueError("歌詞ファイルに有効な行が見つかりません")
print(f"歌詞: {len(lyrics_lines)} 行")

# Whisperモデルでアライメント
import stable_whisper
print(f"Whisperモデル '{MODEL}' を読み込み中...")
model = stable_whisper.load_model(MODEL)

print("アライメント処理中（数分かかることがあります）...")
result = model.align(audio_filename, lyrics_lines, language=LANGUAGE)
segments = [{"start": seg.start, "end": seg.end, "text": seg.text} for seg in result.segments]
segments = fix_overlaps(segments)

# SRTファイルを書き出す
output_path = Path(OUTPUT_FILENAME) if OUTPUT_FILENAME else Path(audio_filename).with_suffix(".srt")
write_srt(segments, output_path)
print(f"\n✓ {len(segments)} 行の字幕を生成: {output_path}")

# ダウンロード
from google.colab import files
files.download(str(output_path))
print("✓ ダウンロード開始")